# EV7 Taxi Customer Cleansing — ผู้เช่าซื้อรถ EV taxi

**ที่มา:** ฐาน `EV7` (`43.254.133.143\LOB_DB_PROD` · profile `ev7`) — ระบบสัญญาเช่าซื้อแท็กซี่ EV + การชำระเงิน + GPS
ลูกค้าอยู่ 2 ตาราง:

| ตาราง | แถว | คอลัมน์ลูกค้า |
|---|---|---|
| `EV7_ContractInformation` | ~2.6k | `CUSTOMER_NAME` (ชื่อเต็มช่องเดียว) · `CITIZEN_ID` · `PHONE` · `LINE_ID` |
| `EV7_CUSTOMER_OPERATION` | ~5.7k | `prefix` · `first_name` · `last_name` · `identitycard` · ที่อยู่ + จังหวัด |

**ทำ:** clean ชื่อ/เบอร์/เลขบัตรด้วยกฎกลาง ([`scripts/ssot/cleanse.py`](../ssot/cleanse.py)) → รวมสองตาราง → เช็กซ้ำ R1 เลขบัตร · R2 ชื่อ+เบอร์ · ชื่อ+เลขบัตรขัดกัน
**ไม่ใช้:** `EV7_SMARTTAXI_TRACKING` (GPS 41M แถว · ไม่มีข้อมูลลูกค้า) · ทะเบียนรถไม่ถูกแสดงหรือบันทึก

In [ ]:
import sys, json, time, collections, os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
sys.path.insert(0, r'C:\Projects\my-first-project\scripts')
import db
from ssot.cleanse import *
pd.set_option('future.infer_string', False)
OUT = r'C:\Projects\my-first-project\scripts\ev7\_out'
cur = db.connect('ev7', timeout=300).cursor()
cur.execute("SELECT CONTRACT_NO, CUSTOMER_NAME, CITIZEN_ID, PHONE, LINE_ID, STATUS_ID, CONTRACT_STRART FROM dbo.EV7_ContractInformation WITH (NOLOCK)")
ci = pd.DataFrame.from_records(cur.fetchall(), columns=[c[0] for c in cur.description])
cur.execute("SELECT id_taxi, prefix, first_name, last_name, identitycard, province, status, contract_date FROM dbo.EV7_CUSTOMER_OPERATION WITH (NOLOCK)")
op = pd.DataFrame.from_records(cur.fetchall(), columns=[c[0] for c in cur.description])
print(f'ContractInformation {len(ci):,} · CUSTOMER_OPERATION {len(op):,}')

## Standardize · Validate (แต่ละตาราง)

In [ ]:
S = pd.concat([
    pd.DataFrame({'src': 'contract', 'src_id': ci.CONTRACT_NO.astype(str), 'title': None, 'first': ci.CUSTOMER_NAME, 'last': '', 'idno': ci.CITIZEN_ID, 'phone': ci.PHONE, 'line': ci.LINE_ID}),
    pd.DataFrame({'src': 'operation', 'src_id': op.id_taxi.astype(str), 'title': op.prefix, 'first': op.first_name, 'last': op.last_name, 'idno': op.identitycard, 'phone': None, 'line': None})],
    ignore_index=True)
nm = clean_names(S['first'], S['last'], S.title.fillna('').astype(str))
ph = clean_phone(S.phone); idd, idv, idk = thai_id(S.idno)
junk = (nm['junk'] != '').to_numpy(); has_id = idk == 1
status = np.select([junk & (ph['status'] != 'ok').to_numpy() & ~has_id, nm['internal'].to_numpy(), nm['juristic'].to_numpy() | (idk == 2), junk], [1, 4, 2, 3], 0).astype('u1')
N = name_hash(nm['key'], junk); ID = id_hash(idd, has_id); P = ph['val']
Q = pd.DataFrame({'src': S.src, 'rows': 1, 'name_ok': ~junk, 'title_removed': nm['title_in'], 'name_split': nm['split'], 'id_present': (idd != ''), 'id_valid_person': has_id,
                  'phone_ok': (ph['status'] == 'ok'), 'mobile': ph['mobile'], 'line_id': S.line.fillna('').astype(str).str.strip() != '', 'person': status == 0})
display(Q.groupby('src').sum().T.style.format('{:,.0f}'))

## Duplicate matching — ภายในตาราง และข้ามสองตาราง

In [ ]:
person = status == 0
rules = [('R1 เลขบัตร', [(ID, ID != 0)]), ('R2 ชื่อ+เบอร์', [(mix(N, P), (N != 0) & (P != 0))]), ('R8 ชื่อ-นามสกุลเดียวกัน (ฐานเล็ก)', [(N, N != 0)])]
steps, n_person, unique, lab = dedup_keys(len(S), person, rules)
display(pd.DataFrame(steps).style.format({'edges': '{:,}', 'merged': '{:,}', 'unique_after': '{:,}'}))
src = S.src.to_numpy()
g = pd.DataFrame({'c': lab[person], 's': src[person]}).groupby('c').s.agg(lambda x: '+'.join(sorted(set(x))))
OV = g.value_counts().to_dict()
print('ลูกค้าไม่ซ้ำ', f'{unique:,}', '· อยู่ในตารางไหน:', OV)
lab1 = lab.astype('u8') + 1
QC = dict(clusters_with_2plus_ids=int(len(np.unique(lab1[distinct_per_key(lab1, ID, person & (ID != 0)) >= 2]))),
          ids_with_2plus_names=int(len(np.unique(ID[distinct_per_key(ID, N, person & (ID != 0) & (N != 0)) >= 2]))))
QC

**R8 ชื่อ-นามสกุลเดียวกัน** ใช้เฉพาะ EV7 (ฐานเล็ก ~5k คน · สัญญาส่วนใหญ่ไม่มีเลขบัตร/เบอร์) — ตรวจผลด้วย `clusters_with_2plus_ids` (กลุ่มที่รวมแล้วเลขบัตรขัดกัน) · ในฐานใหญ่ (CRM/ITEC) ห้ามใช้เพราะชื่อซ้ำกันเยอะ

In [ ]:
a = dict(status=status, cluster=lab)
RD = {'national_id': cluster_any(a, ID != 0), 'mobile': cluster_any(a, ph['mobile']), 'line_id': cluster_any(a, (S.line.fillna('').astype(str).str.strip() != '').to_numpy())}
cnt = {f'status_{k}': int(v) for k, v in zip(*np.unique(status, return_counts=True))}
cnt.update({k: int(v) for k, v in Q.drop(columns='src').sum().items()})
cnt.update(rows_contract=int((src == 'contract').sum()), rows_operation=int((src == 'operation').sum()))
f = [('แถวดิบ (2 ตาราง)', len(S))] + [(f'✂ status {k}', -v) for k, v in cnt.items() if k.startswith('status_') and k != 'status_0'] + [('✂ ' + s['rule'], -s['merged']) for s in steps] + [('= ลูกค้าไม่ซ้ำ', unique)]
for l_, v in f: print(f'   {l_:<40}{v:>8,}')
SUMMARY = {'EV7': dict(name='EV7 Taxi · ผู้เช่าซื้อ', funnel=f, cnt=cnt, steps=steps, n_person=n_person, unique=unique, overlap=OV, readiness=RD, qc=QC), '_run': time.strftime('%Y-%m-%d %H:%M')}
os.makedirs(OUT, exist_ok=True)
json.dump(SUMMARY, open(os.path.join(OUT, 'ev7_cleansing_summary.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=int)
print(RD)